> ## ⚠️ Before you run anything: switch this notebook to a **GPU runtime**
> In Colab: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. The setup cell
> checks for a GPU and stops if there is none. Without one the finetuning stages and the LAMMPS run
> are not feasible, and even the cached pass through the notebook becomes much slower.

# GRACE Al–Li tutorial — finetuning, uncertainty, distillation and MD

Finetune the `GRACE-3L-OMAT-large` foundation model on the Al–Li convex hull **plus 128 further DFT
structures that the foundation model itself picks** out of 1000 candidates by farthest-point
sampling in its own feature space; build uncertainty artifacts; distil into a fast GRACE/FS
student; run LAMMPS MD with an on-the-fly extrapolation grade and select the structures worth
computing next. Results of the same pipeline trained on the hull set alone, the **hull-only
reference**, are shown wherever the added data makes a difference.

<img src="https://raw.githubusercontent.com/yury-lysogorskiy/grace-colab-tutorial/main/figures/tutorial-scheme.png" alt="the tutorial on one page" width="100%">

*The tutorial on one page. Colours: blue = models, amber = DFT data, violet = uncertainty and
selection tools, green = simulation; the tag on each box is the section where it is built.*

Every expensive stage writes its result to disk and is skipped when that result already exists.
The repository ships those results, so a first pass through the notebook on a Colab T4 takes about
15 minutes; set `FORCE_RERUN = True` in §0 to recompute everything, about an hour.

| § | stage |
|---|---|
| 1 | Data: the convex-hull training set and the candidate pool |
| 2 | Baseline: what the foundation model predicts untouched |
| 3 | **Selecting what to compute: `grace_uq select --strategy fps-all`** |
| 4 | Finetune the teacher `GRACE-3L-OMAT-large` on hull + selection |
| 5 | Uncertainty artifacts with `grace_uq` |
| 6 | Teacher validation, against the hull-only reference |
| 7 | UQ plots and the out-of-distribution demo |
| 8 | The convex hull before and after finetuning; does γ predict error? |
| 9 | Distillation pool |
| 10 | Distil into a `GRACE-FS-OMAT` student |
| 11 | Convex hull from relaxed structures: DFT vs teacher vs student |
| 12 | Validation beyond the hull: energy–volume curves, elastic constants, phonons |
| 13 | LAMMPS MD with extrapolation grade, and `pace_select` |

## Setup

On Colab this cell installs the GRACE tools, `amstools` for the convex-hull bookkeeping, the
GRACE/FS C++ evaluator from `python-ace` (a source build, a few minutes), and clones the tutorial
repository with its cached results. The LAMMPS binary is only needed to *re-run* the MD in §13;
the shipped MD results cover the section otherwise.

In [ ]:
import os, subprocess, sys

REPO_URL     = "https://github.com/yury-lysogorskiy/grace-colab-tutorial"
LMP_DRIVE_ID = ""          # Google Drive id of lmp-grace-t4-cu128.tar.gz (LAMMPS with GRACE/FS and Kokkos/CUDA for the T4); empty = skip
IN_COLAB     = "google.colab" in sys.modules

def sh(cmd):
    """Run a shell command and stream its output into the cell."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {cmd}")

if IN_COLAB:
    if subprocess.run("nvidia-smi -L", shell=True, capture_output=True).returncode != 0:
        raise RuntimeError("No GPU in this runtime. Runtime -> Change runtime type -> Hardware accelerator: T4 GPU, then run this cell again.")
    if not os.path.exists("/content/grace-colab-tutorial"):
        sh(f"git clone --depth 1 {REPO_URL} /content/grace-colab-tutorial")
    sh("apt-get install -y -qq libopenmpi3 > /dev/null")                  # libmpi.so.40 for the LAMMPS binary
    sh("pip install -q tensorpotential==0.6.1 amstools 'phonopy<4' gdown cmake")   # amstools 0.4.1 needs phonopy 2.x
    if not os.path.exists("/content/python-ace"):                         # GRACE/FS evaluator, pace_activeset, pace_select:
        sh("git clone -q -b feature/grace_fs https://github.com/ICAMS/python-ace.git /content/python-ace")
        sh("CMAKE_BUILD_PARALLEL_LEVEL=2 pip install -v /content/python-ace 2>&1 | grep --line-buffered -E '\\[ *[0-9]+%\\]|Successfully|error'")   # a source build of a few minutes; -v shows CMake's progress
    if LMP_DRIVE_ID and not os.path.exists("/content/lmp-grace/bin/lmp"):
        sh(f"gdown {LMP_DRIVE_ID} -O /content/lmp-grace.tar.gz && tar xzf /content/lmp-grace.tar.gz -C /content")

## 0. Configuration

The knobs and paths. Everything else in the notebook reads from here.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/content/grace-colab-tutorial") if "google.colab" in sys.modules else Path.cwd()   # the cloned repository

# ---- knobs ----------------------------------------------------------------------
FORCE_RERUN    = False      # True = recompute every stage even if its output exists
XLA_CACHE_DIR  = None       # e.g. ROOT / ".xla-cache": persistent XLA kernel cache shared by every process here;
                            # about halves a repeated compilation, but the first compilation of a shape gets slower
N_UPDATES      = 250        # teacher: total gradient updates
RP_DIM         = 128        # UQ: random-projection dimension (the shipped UQ artifacts were built with 128)
N_CLUSTERS     = "1 2 4 8"  # UQ: GMM cluster counts tried by the elbow search
UQ_PCTL        = 99         # UQ: calibrate gamma so ~1% of TRAINING atoms read gamma > 1
POOL_NVOL      = 7          # distillation pool: volume points per seed structure
POOL_NRAND     = 7          # distillation pool: strain+rattle samples per seed structure
POOL_GAMMA_MAX = 5          # distil only from pool structures with teacher gamma <= this; None = whole pool
FS_ITERS       = 30         # student: BFGS iterations (the shipped student used 50)
MD_STEPS       = 1500       # LAMMPS: MD steps
TEACHER_NAME   = "GRACE-3L-hull+fps-ft"   # label of the finetuned teacher in tables and plots
N_SELECT       = 128        # data selection: structures picked from the candidate pool
FPS_SEED       = 0          # data selection: seed of the farthest-point sampling

# ---- inputs, shipped with the tutorial -------------------------------------------
DATA        = ROOT / "0-data"
TRAIN_SET   = DATA / "AlLi-hull-train.pkl.gz"       # 112 hull training structures
TEST_SET    = DATA / "AlLi-hull-test.pkl.gz"        # 16 held-out structures, the same 16 as in the hull-only reference
OOD_SET     = DATA / "AlLi-ood-mp1191737.pkl.gz"    # 32 structures of a prototype never shown to the model
EXT_SET     = DATA / "AlLi-extended-eval.pkl.gz"    # 265 further DFT structures for the final error table
MP_SET      = DATA / "MP-hull-candidates.pkl.gz"    # 19 Materials Project Al-Li structures to relax
CANDIDATES  = DATA / "AlLi-candidates-1000.pkl.gz"  # 1000 random DFT structures to select from
REF_ERRORS  = DATA / "reference-hull-only-errors.pkl.gz"    # results of the hull-only reference run, for comparison
REF_SUMMARY = DATA / "reference-hull-only-summary.pkl.gz"

# ---- outputs, one directory per stage --------------------------------------------
OUT     = ROOT                                   # where this notebook writes its stages
FMDIR   = OUT / "1-foundation-baseline"
SELECT  = OUT / "1-select"
TEACHER = OUT / "1-finetune-hull+fps"
UQDIR   = OUT / "2-uq-validation"
DISTILL = OUT / "3-distill"
HULLDIR = OUT / "4-convex-hull"
TSEED   = TEACHER / "seed" / "1"                 # where gracemaker writes the teacher
UQART   = TSEED / f"gmm_artifacts_rp{RP_DIM}_p{UQ_PCTL}.npz"     # UQ artifact and the SavedModel that also returns gamma:
UQMODEL = TSEED / f"saved_model_uq_rp{RP_DIM}_p{UQ_PCTL}"        # the UQ settings are part of the names
# the student and everything after it depend on which part of the pool it was trained on, so the
# pool choice is part of these names: a changed POOL_GAMMA_MAX trains a new student, never reuses a stale one
POOL_TAG = "full-pool" if POOL_GAMMA_MAX is None else f"gamma-le-{POOL_GAMMA_MAX:g}"
STUDENT = DISTILL / f"student-{POOL_TAG}"        # where gracemaker is started for the student
SSEED   = STUDENT / "seed" / "1"                 # where gracemaker writes the student
MD      = OUT / f"5-lammps-{POOL_TAG}"           # the MD run uses that student

# ---- LAMMPS ------------------------------------------------------------------------
LMP = Path("/content/lmp-grace/bin/lmp")         # LAMMPS with GRACE/FS and Kokkos/CUDA, unpacked by the setup cell; running locally: point this to your own Kokkos build

### Helpers

Small functions used by several sections: running the command-line tools, loading data,
evaluating a model on a set of structures, and the two plots that repeat (learning curves and
parity). Nothing here is specific to Al–Li; skim it and move on.

In [ ]:
import os, sys, time, shutil, subprocess, signal, logging, yaml
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
logging.disable(logging.INFO)                          # quieten the INFO chatter of the libraries

# The GRACE tools and LAMMPS run as subprocesses of this kernel. Give them the kernel's
# environment (its console scripts and libraries) and let them share the GPU.
BIN = Path(sys.executable).parent
os.environ["PATH"]                     = f"{BIN}:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"]          = f"{BIN.parent}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CUDA_VISIBLE_DEVICES"]     = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["PYTHONUNBUFFERED"]         = "1"
if XLA_CACHE_DIR is not None:                          # must be set before TensorFlow is imported; inherited by the tools
    Path(XLA_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    os.environ["XLA_FLAGS"] = (f"--xla_gpu_kernel_cache_file={Path(XLA_CACHE_DIR) / 'kernels.bin'} "
                               f"--xla_gpu_enable_llvm_module_compilation_parallelism=true " + os.environ.get("XLA_FLAGS", ""))

def run(cmd, cwd=ROOT, **env):
    """Run a shell command, stream its output into the notebook, raise if it fails.
    Interrupting the cell (the stop button) stops the command as well."""
    print(f"$ {cmd}")
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **env}, start_new_session=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        os.killpg(proc.pid, signal.SIGINT)                 # forward the interrupt to the command and its children
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
        raise
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

def have(path, what="", fixed=False):
    """True if `path` exists and FORCE_RERUN is off, i.e. the stage that produces it can be skipped.
    fixed=True marks a shipped result that FORCE_RERUN must not recompute."""
    path = Path(path)
    ok = path.exists() and (fixed or not FORCE_RERUN)
    if ok: print(f"[cached] {path.relative_to(ROOT)} {what}")
    return ok

def load(path):     return pd.read_pickle(path, compression="gzip")
def save(df, path): df.to_pickle(path, compression="gzip")
def natoms(df):     return df["ase_atoms"].map(len).values
def fmax(df):       return df["forces"].map(lambda f: np.abs(np.asarray(f)).max()).values  # largest force component per structure
def flat(forces):   return np.concatenate([np.asarray(f).ravel() for f in forces])         # all force components in one array

# ---- model evaluation ----------------------------------------------------------------
from tensorpotential.calculator import TPCalculator, grace_fm, predict_structures

def predict(atoms, calc):
    """Energies and forces of a list of structures, in the given order.
    predict_structures evaluates the largest structure first so one compiled shape covers the rest."""
    out = predict_structures(list(atoms), calc, properties=("energy", "forces"))
    return np.array(out["energy"]), out["forces"]

def errors(E_ref, E_pred, F_ref, F_pred, nat):
    """Energy MAE in meV/atom and force-component MAE in meV/A."""
    return (np.abs(E_pred - E_ref)/nat).mean()*1000, np.abs(flat(F_pred) - flat(F_ref)).mean()*1000

def parity_plot(E_ref, E_pred, F_ref, F_pred, nat, ref="DFT", model="GRACE", color="tab:blue", title=None):
    """Energy-per-atom and force parity plots; returns the two MAEs."""
    e_mae, f_mae = errors(E_ref, E_pred, F_ref, F_pred, nat)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].scatter(E_ref/nat, E_pred/nat, s=28, color=color)
    ax[0].set_title(f"energy parity - MAE {e_mae:.2f} meV/atom")
    ax[1].scatter(flat(F_ref), flat(F_pred), s=5, alpha=.4, color=color)
    ax[1].set_title(f"force parity - MAE {f_mae:.2f} meV/A")
    for a, unit in zip(ax, ["E/atom (eV)", "force (eV/A)"]):
        lim = [min(a.get_xlim()[0], a.get_ylim()[0]), max(a.get_xlim()[1], a.get_ylim()[1])]
        a.plot(lim, lim, "k--", lw=1)
        a.set_xlabel(f"{ref} {unit}"); a.set_ylabel(f"{model} {unit}")
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()
    return e_mae, f_mae

def learning_curves(seed_dir, title, xlabel="epoch"):
    """Train/test energy and force MAE per epoch from gracemaker's metrics files; returns the final values."""
    metrics = {s: pd.json_normalize(yaml.safe_load(open(seed_dir/f"{s}_metrics.yaml"))) for s in ("train", "test")}
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a, col, lab in zip(ax, ["mae/depa", "mae/f_comp"], ["energy MAE (eV/atom)", "force MAE (eV/A)"]):
        for s, m in metrics.items(): a.plot(m["epoch"], m[col], label=s)
        a.set_yscale("log"); a.set_xlabel(xlabel); a.set_ylabel(lab); a.legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()
    final = pd.DataFrame({s: m.iloc[-1][["mae/depa", "mae/f_comp"]].values*1000 for s, m in metrics.items()},
                         index=["E-MAE (meV/atom)", "F-MAE (meV/A)"])
    return final.round(2)

missing = [t for t in ("gracemaker", "grace_uq", "grace_predict", "grace_utils", "pace_activeset", "pace_select")
           if shutil.which(t) is None]
if missing:
    raise RuntimeError(f"GRACE command-line tools not found: {missing}. Start the kernel from the environment that provides them.")
print("environment :", BIN.parent)
print("LAMMPS      :", LMP if LMP.exists() else "not downloaded (set LMP_DRIVE_ID in the setup cell); the shipped MD results are used")

## 1. Data — the convex-hull training set

The dataset is **128 structures**: the 7 Al–Li convex-hull prototypes, each with its relaxed cell
(`opt`), a Murnaghan volume scan (`murn`) and phonon-displaced supercells (`phon`).

Why those three calculation types: `opt` and `murn` are relaxed or symmetric, so their forces
essentially vanish; `phon` supplies the only real force information. Nearest-neighbour distance
scans (`nn`) are deliberately excluded; they contain isolated atoms beyond the 6 Å model cutoff and
multi-eV outliers.

The train/test split is **preselected and shipped in `0-data/`** (112 + 16) rather than drawn
randomly at fit time, so the held-out set is identical for everyone running this notebook. A third
file holds 32 structures of one more prototype, `LiAl_mp-1191737`, which is never fitted and serves
as the out-of-distribution test in §7.

The dataset has **no stress**, so stress is dropped from the loss throughout.

All DFT data in this tutorial are taken from the Al–Li database of Menon, Lysogorskiy, Knoll,
Leimeroth, Poul, Qamar, Janssen, Mrovec, Rohrer, Albe, Behler, Drautz and Neugebauer, *From
electrons to phase diagrams with machine learning potentials using pyiron based automated
workflows*, npj Comput. Mater. **10**, 261 (2024),
[doi:10.1038/s41524-024-01441-0](https://www.nature.com/articles/s41524-024-01441-0).

In [ ]:
train, test, ood = load(TRAIN_SET), load(TEST_SET), load(OOD_SET)
train["split"], test["split"], ood["split"] = "train", "test", "OOD"
ds = pd.concat([train, test, ood], ignore_index=True)
ds["nat"]   = natoms(ds)
ds["fmax"]  = fmax(ds)
ds["proto"] = ds["name"].astype(str).str.extract(r"/DFT/([^/]+)/")[0]        # hull prototype
ds["kind"]  = ds["name"].astype(str).str.extract(r"/DFT/[^/]+/([^/]+)/")[0]   # opt / murn / phon
fitted = (ds["split"] != "OOD").values                                        # what the teacher will see

print(f"train {len(train)} + test {len(test)} = {fitted.sum()} fitted structures, {ds.loc[fitted, 'nat'].sum()} atoms")
print(f"plus {len(ood)} OOD structures (LiAl_mp-1191737), never used for fitting")
display(pd.crosstab(ds.loc[fitted, "proto"], ds.loc[fitted, "kind"]))
print("\n|F|max by calculation type, fitted data only (eV/A):")
display(ds[fitted].groupby("kind")["fmax"].agg(["median", "max"]).round(4))

### Convex hull of the tutorial dataset

The hull is built from **this dataset alone**. `with_hull` adds composition, formation energy and
distance to the hull to a dataframe (using `amstools.thermodynamics`), and `lower_hull` returns the
hull vertices. Both are reused every time a model's hull is compared with DFT later on.

In [ ]:
from amstools.thermodynamics import (ensure_energy_per_atom_column, compute_compositions,
                                     compute_formation_energy, compute_convexhull_dist)

HULL_COLS = ["energy_per_atom", "e_formation_per_atom", "e_chull_dist_per_atom"]

def with_hull(df):
    """Copy of df with composition, formation energy and distance to the convex hull,
    all derived from its `energy` column."""
    df = df.drop(columns=HULL_COLS, errors="ignore").copy()      # always recompute from `energy`
    ensure_energy_per_atom_column(df, energy_column="energy")
    compute_compositions(df)
    compute_formation_energy(df, verbose=False)
    compute_convexhull_dist(df, verbose=False)
    return df

def lower_hull(df):
    """Vertices (c_Li, Ef) of the convex hull spanned by the structures in df: the ones amstools
    puts at zero hull distance, lowest per composition."""
    d  = with_hull(df)                                            # the hull of THESE structures only
    on = d[d["e_chull_dist_per_atom"] < 1e-6]
    pts = on.groupby(on["c_Li"].round(3))["e_formation_per_atom"].min()
    return sorted(zip(pts.index, pts.values))

def hull_at(hull, c):
    """Hull energy at composition c, interpolated between vertices."""
    return np.interp(c, [x for x, _ in hull], [y for _, y in hull])

def plot_hull(ax, hull, *fmt, **style):
    ax.plot([c for c, _ in hull], [e*1000 for _, e in hull], *fmt, **style)

ds   = with_hull(ds)                    # formation energies for all 160 structures ...
hull = lower_hull(ds[fitted])           # ... but the hull from the 128 fitted ones only

display(pd.DataFrame({"c_Li": [c for c, _ in hull], "Ef (meV/atom)": [round(e*1000, 1) for _, e in hull]}))
d_hull = ds["e_chull_dist_per_atom"]*1000
print(f"on the hull (< 1 meV/atom): {(d_hull < 1).sum()} structures")
print(f"OOD sits {d_hull[~fitted].min():.0f}-{d_hull[~fitted].max():.0f} meV/atom above the hull")

fig, ax = plt.subplots(figsize=(8, 5))
for split, st in {"train": dict(color="tab:blue",   s=34, marker="o"),
                  "test":  dict(color="tab:red",    s=62, marker="o"),
                  "OOD":   dict(color="tab:orange", s=62, marker="^")}.items():
    sub = ds[ds["split"] == split]
    ax.scatter(sub["c_Li"], sub["e_formation_per_atom"]*1000, alpha=.85, zorder=3,
               label=f"{split} ({len(sub)})" + (" - held out" if split == "OOD" else ""), **st)
plot_hull(ax, hull, "k.-", lw=1.5, ms=11, zorder=1, label="convex hull")
ax.set_xlabel("$c_{Li}$"); ax.set_ylabel("formation energy (meV/atom)")
ax.set_title("Al-Li convex hull: fitted data and the held-out prototype")
ax.legend(); plt.tight_layout(); plt.show()

### The candidate pool

Next to the hull set, this tutorial ships **1000 further DFT structures** drawn at random from the
same Al–Li database (Menon *et al.*, npj Comput. Mater. **10**, 261 (2024), see §1) after the same
hygiene filters as the evaluation set (no nearest-neighbour
scans, no overlapping or isolated atoms, at most 500 meV/atom above the hull, 2 to 16 atoms), with
everything used elsewhere in this notebook removed. Think of them as "DFT we could add"; §3 decides which
128 of them are worth adding. The draw is deliberately uniform, so like the database itself it is
about 60 % pure Al and pure Li.

In [ ]:
candidates = load(CANDIDATES)
candidates["group"] = np.where(candidates["kind"] == "generated", "generated cell", "prototype deformation")
candidates["comp"]  = pd.cut(candidates["c_Li"], [-0.01, 0.001, 0.999, 1.0], labels=["pure Al", "Al-Li alloy", "pure Li"])
display(pd.crosstab(candidates["group"], candidates["comp"], margins=True))
print(f"{len(candidates)} candidates, {candidates['nat'].sum()} atoms, {candidates['nat'].min()}-{candidates['nat'].max()} atoms per cell")
print(f"|F|max: median {candidates['fmax'].median():.2f}, max {candidates['fmax'].max():.1f} eV/A  (hull set median {np.median(ds.loc[fitted, 'fmax']):.3f})")

# formation energies of the candidates on the same pure-element reference as the hull set
COLS = ["name", "ase_atoms", "energy", "forces"]
cand_hull = with_hull(pd.concat([ds.loc[fitted, COLS].assign(group="hull set"),
                                 candidates[COLS + ["group"]]], ignore_index=True))
cand_hull = cand_hull[cand_hull["group"] != "hull set"]            # the candidate rows, in order
candidates["e_formation_per_atom"]  = cand_hull["e_formation_per_atom"].values
candidates["e_chull_dist_per_atom"] = cand_hull["e_chull_dist_per_atom"].values

fig, ax = plt.subplots(figsize=(8.5, 5))
for grp, color in [("generated cell", "0.7"), ("prototype deformation", "tab:purple")]:
    sub = candidates[candidates["group"] == grp]
    ax.scatter(sub["c_Li"], sub["e_formation_per_atom"]*1000, s=10, alpha=.6, color=color, zorder=2, label=f"{grp} ({len(sub)})")
ax.scatter(ds.loc[fitted, "c_Li"], ds.loc[fitted, "e_formation_per_atom"]*1000, s=26, color="tab:blue", zorder=3, label=f"hull set ({fitted.sum()})")
plot_hull(ax, hull, "k.-", lw=1.5, ms=9, zorder=4, label="DFT hull")
ax.set_xlabel("$c_{Li}$"); ax.set_ylabel("formation energy (meV/atom)")
ax.set_title("The candidate pool: DFT structures the hull set does not contain")
ax.legend(); plt.tight_layout(); plt.show()

## 2. Baseline — what `GRACE-3L-OMAT-large` predicts untouched

Before training anything, it is worth asking what the foundation model already knows about Al–Li.
It has never seen this dataset, but it was trained on OMat24, which covers a great deal of
inorganic chemistry.

Evaluate it on the same structures and rebuild the hull from **its own** predicted energies.
Formation energies are referenced to its own pure Al and pure Li, so the absolute-energy offset
between OMat24's DFT setup and this tutorial's cancels and only the *shape* of the hull is compared.
That is the only fair way to compare two models trained against different references, and the same
recipe is used for every model hull in this notebook.

This is the baseline every later section is measured against. The foundation-model calculator stays
loaded; the error table in §9 uses it again.

In [ ]:
FMDIR.mkdir(exist_ok=True)
fm_file = FMDIR / "fm_pred.pkl.gz"
calc_fm = grace_fm("GRACE-3L-OMAT-large")              # kept for the error table in section 9
if not have(fm_file, "(foundation-model energies)"):
    t0 = time.time()
    E, _ = predict(ds["ase_atoms"], calc_fm)
    save(pd.DataFrame({"name": ds["name"], "E_fm": E}), fm_file)
    print(f"evaluated {len(ds)} structures in {time.time()-t0:.0f}s")
fm = load(fm_file)
assert (fm["name"].values == ds["name"].values).all(), "prediction rows out of order"

ds_fm   = with_hull(ds.assign(energy=fm["E_fm"].values))     # hull from the foundation model's OWN energies
hull_fm = lower_hull(ds_fm[fitted])

def hull_table(model_hulls):
    """Hull energy (meV/atom) of each model at the DFT hull compositions, and its difference to DFT."""
    t = pd.DataFrame({"c_Li": [c for c, _ in hull], "DFT": [e*1000 for _, e in hull]}).set_index("c_Li")
    for name, h in model_hulls.items():
        t[name] = hull_at(h, t.index)*1000
        t[f"{name} - DFT"] = t[name] - t["DFT"]
    return t.round(1)

def formation_mae(model_ds):
    """Mean and largest |formation-energy error| vs DFT over the fitted structures (meV/atom)."""
    err = np.abs(model_ds.loc[fitted, "e_formation_per_atom"] - ds.loc[fitted, "e_formation_per_atom"])*1000
    return err.mean(), err.max()

display(hull_table({"foundation": hull_fm}))
mae, worst = formation_mae(ds_fm)
print(f"formation-energy MAE vs DFT: {mae:.1f} meV/atom (max {worst:.1f}) over {fitted.sum()} structures")
print(f"hull minimum: DFT {min(e for _, e in hull)*1000:.1f} vs foundation {min(e for _, e in hull_fm)*1000:.1f} meV/atom")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.scatter(ds.loc[fitted, "c_Li"], ds.loc[fitted, "e_formation_per_atom"]*1000, s=18, color="0.8", zorder=1, label="DFT structures")
plot_hull(ax, hull,    "k.-", lw=2.2, ms=9, zorder=3, label="DFT hull (reference)")
plot_hull(ax, hull_fm, "s--", color="tab:orange", lw=1.8, ms=6, zorder=2, label="GRACE-3L-OMAT-large (no finetuning)")
ax.set_xlabel("$c_{Li}$"); ax.set_ylabel("formation energy (meV/atom)")
ax.set_title("Al-Li convex hull straight out of the foundation model")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Selecting what to compute — `grace_uq select`

The hull set is a physicist's choice. The candidate pool is what a database offers. Which 128 of
the 1000 are worth adding? Ask the model that will be finetuned: `GRACE-3L-OMAT-large` turns every
atom into a 132-dimensional feature vector, and **farthest-point sampling** (`fps-all`) in that
space picks structures that are as different from each other as possible, whatever their
composition or energy. No training has happened yet, so the choice is the foundation model's alone.

Two commands. `grace_uq predict --save-features` evaluates the pool and stores the per-atom
features; `grace_uq select --strategy fps-all` picks from them, keeping at least D+1 atoms of every
element so the selection cannot drift to one species. The same tool offers `random-all` as the
baseline, and `fps-extrap` / `random-extrap`, which first filter by γ once a finetuned model exists;
those are the active-learning strategies applied to the MD trajectory in §13.

The selection is shipped with the tutorial and is **never recomputed, not even with `FORCE_RERUN`**:
features differ in the last digits between GPUs, so a re-run could pick a slightly different 128 and
every later stage would change with it. To select afresh, delete the files in `1-select/`.

In [ ]:
from tensorpotential.calculator.foundation_models import get_or_download_model

FM_DIR = Path(get_or_download_model("GRACE-3L-OMAT-large"))     # local copy of the foundation model
SELECT.mkdir(parents=True, exist_ok=True)
predicted_file = SELECT / "candidates_predicted.pkl.gz"
hull_pred_file = SELECT / "hull_predicted.pkl.gz"
selected_file  = SELECT / f"selected_fps{N_SELECT}.pkl.gz"

# fixed=True: the shipped selection is never recomputed, not even with FORCE_RERUN (see the text above)
if not have(predicted_file, "(candidate features)", fixed=True):
    run(f"grace_uq predict --model {FM_DIR} --dataset {CANDIDATES} --output {predicted_file} --save-features --n-workers 1")
if not have(hull_pred_file, "(hull-set features, for the plots only)", fixed=True):
    run(f"grace_uq predict --model {FM_DIR} --dataset {DATA/'AlLi-hull-all.pkl.gz'} --output {hull_pred_file} --save-features --n-workers 1")
if not have(selected_file, "(fps selection)", fixed=True):
    run(f"grace_uq select --model {FM_DIR} --predicted {predicted_file} -n {N_SELECT} --strategy fps-all --seed {FPS_SEED} --output {selected_file}")

selected = load(selected_file)
candidates["selected"] = candidates["name"].isin(selected["name"])
assert candidates["selected"].sum() == len(selected) == N_SELECT
print(f"selected {len(selected)} of {len(candidates)} candidates, {natoms(selected).sum()} atoms")
display(pd.crosstab(candidates["group"], candidates["comp"], values=candidates["selected"], aggfunc="sum", margins=True).astype(int))

In [ ]:
from sklearn.decomposition import PCA

def mean_features(predicted):
    """One feature vector per structure: the mean over its atoms of the per-atom features."""
    return np.stack([np.asarray(f, float).mean(axis=0) for f in predicted["features"]])

X_cand, X_hull = mean_features(load(predicted_file)), mean_features(load(hull_pred_file))
pca = PCA(n_components=2).fit(np.vstack([X_cand, X_hull]))           # the two leading directions of the feature space
P_cand, P_hull = pca.transform(X_cand), pca.transform(X_hull)
explained = 100*pca.explained_variance_ratio_
sel = candidates["selected"].values

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
ax[0].scatter(candidates.loc[~sel, "c_Li"], candidates.loc[~sel, "e_formation_per_atom"]*1000, s=9,  color="0.75",    label=f"not selected ({(~sel).sum()})")
ax[0].scatter(candidates.loc[sel,  "c_Li"], candidates.loc[sel,  "e_formation_per_atom"]*1000, s=26, color="tab:red", label=f"selected ({sel.sum()})", zorder=3)
ax[0].scatter(ds.loc[fitted, "c_Li"], ds.loc[fitted, "e_formation_per_atom"]*1000, s=22, marker="x", color="tab:blue", label="hull set", zorder=2)
plot_hull(ax[0], hull, "k-", lw=1.2, zorder=1)
ax[0].set_xlabel("$c_{Li}$"); ax[0].set_ylabel("formation energy (meV/atom)"); ax[0].set_title("energy view"); ax[0].legend(fontsize=8)

ax[1].scatter(P_cand[~sel, 0], P_cand[~sel, 1], s=9,  color="0.75",    label="not selected")
ax[1].scatter(P_cand[sel, 0],  P_cand[sel, 1],  s=26, color="tab:red", label="selected", zorder=3)
ax[1].scatter(P_hull[:, 0],    P_hull[:, 1],    s=22, marker="x", color="tab:blue", label="hull set", zorder=2)
ax[1].set_xlabel(f"PC 1 ({explained[0]:.0f} % of the variance)"); ax[1].set_ylabel(f"PC 2 ({explained[1]:.0f} %)")
ax[1].set_title("foundation-model feature space (PCA)"); ax[1].legend(fontsize=8)

bins = np.logspace(-3, 1, 40)
for lab, f, color in [("candidates", candidates["fmax"], "0.6"), ("selected", candidates.loc[sel, "fmax"], "tab:red"),
                      ("hull set", ds.loc[fitted, "fmax"], "tab:blue")]:
    f = np.clip(f, 1e-3, None)
    ax[2].hist(f, bins=bins, weights=np.ones(len(f))/len(f), alpha=.5, color=color, label=f"{lab} (median {np.median(f):.2f})")
ax[2].set_xscale("log"); ax[2].set_xlabel("|F|max per structure (eV/A)"); ax[2].set_ylabel("fraction per bin")
ax[2].set_title("force information"); ax[2].legend(fontsize=8)
fig.suptitle(f"fps-all: {N_SELECT} structures as far apart as possible in the foundation model's feature space")
plt.tight_layout(); plt.show()

### The training set: hull + selection

The 128 selected structures join the 112 hull training structures. The 16 hull test structures stay
the test set, identical to the hull-only reference run, so both teachers are scored on the same held-out
data and on the same extended evaluation set later on.

In [ ]:
train_fps  = pd.concat([train[COLS], candidates.loc[sel, COLS]], ignore_index=True)     # 112 hull + 128 selected
train_file = OUT / "train-hull+fps.pkl.gz"
save(train_fps, train_file)

def describe(df):
    c_li = pd.Series([a.get_chemical_symbols().count("Li")/len(a) for a in df["ase_atoms"]]).round(3)
    return {"structures": len(df), "atoms": natoms(df).sum(), "compositions": c_li.nunique(),
            "|F|max median (eV/A)": np.median(fmax(df)), "|F|max max (eV/A)": fmax(df).max()}
display(pd.DataFrame({"hull train": describe(train), "fps selected": describe(candidates[sel]),
                      "combined": describe(train_fps)}).T.round(3))
print(f"the teacher trains on {train_file.relative_to(ROOT)}; the test set stays the {len(test)} hull test structures")

## 4. Finetune the teacher — `GRACE-3L-OMAT-large` on hull + selection

Frozen-readout finetuning: only the readout weights (`reducing_`) are trained, everything else keeps
its foundation-model value. The list of trainable variables in `input.yaml` is specific to the
3-layer model. The recipe is that of the hull-only reference run; only the training file differs.

The input file is written out in full below so that the whole recipe is visible. For a fit of your
own you do not have to type it: `gracemaker -t` builds an `input.yaml` interactively, asking for the
dataset, the foundation model to start from and the fitting options one question at a time.

In [ ]:
from string import Template

TEACHER.mkdir(exist_ok=True)
(TEACHER / "input.yaml").write_text(Template("""
seed: 1
cutoff: n/a

data:
  filename: $TRAIN_FILE
  test_filename: $TEST_FILE
  reference_energy: 0
  save_dataset: True

potential:
  finetune_foundation_model: GRACE-3L-OMAT-large
  reduce_elements: True
  shift: auto

fit:
  # no stress term: the dataset has none
  loss:
    energy: {weight: 16, type: huber, delta: 0.01}
    forces: {weight: 32, type: huber, delta: 0.01}
    switch:                   # after 75% of the updates: lower the learning rate, weight energy up
      after_iter: 0.75
      learning_rate_reduction_factor: 0.1
      energy: {weight: 128}
      forces: {weight: 32}
  target_total_updates: $N_UPDATES
  optimizer: Adam
  opt_params: {learning_rate: 0.001, use_ema: True, ema_momentum: 0.99, weight_decay: null, clipnorm: 1.0}
  scheduler: reduce_on_plateau
  scheduler_params: {patience: 50, reduction_factor: 0.8, minimum_learning_rate: 8.33e-05, stop_at_min: True}
  compute_convex_hull: False
  batch_size: 4
  test_batch_size: 16
  jit_compile: True
  eval_init_stats: True
  # frozen readout: only these variables train (the names are specific to the 3-layer model)
  trainable_variable_names: ["rho1/reducing_", "rho2/reducing_", "rho3/reducing_", "eq1/reducing_", "eq2/reducing_"]
  train_max_n_buckets: 2
  test_max_n_buckets: 1
  checkpoint_freq: 5
  progressbar: False
""").substitute(N_UPDATES=N_UPDATES, TRAIN_FILE=train_file, TEST_FILE=TEST_SET))
print("written", TEACHER / "input.yaml")

In [ ]:
if not (have(TSEED / "final_model", "(teacher already trained)") or have(UQMODEL, "(teacher present as its UQ SavedModel)")):
    t0 = time.time()
    run("gracemaker input.yaml", cwd=TEACHER)
    print(f"\nteacher finetune took {time.time()-t0:.0f}s")

## 5. Uncertainty artifacts — `grace_uq`

GMM-based uncertainty on random-projected invariant features of the teacher. Three settings matter:

* **`--rp-dim 128`**, the default, set by `RP_DIM` in §0: the dimension of the random projection of
  the invariant features that the GMM works in. A smaller value gives a smaller artifact and a lower
  minimum-atoms-per-cluster floor (D+1); with the strained cells in this training set the full feature
  space is kept. The artifact names carry the value, so changing it builds a new UQ model instead of
  reusing the old one.
* **an elbow search over the cluster count**, not a single cluster. One Gaussian cannot cover 7
  distinct prototypes; forcing one roughly triples the fraction of held-out atoms that look
  extrapolative.
* **`--threshold-percentile 99`** calibrates γ so that γ = 1 sits at the 99th percentile of the
  training atoms: about 1% of training atoms read as extrapolative, by construction.

The build also exports a SavedModel that returns γ alongside energy, forces and stress. Its energies,
forces and stress are bit for bit those of `final_model`, so it is loaded once here and serves as
*the* teacher for every evaluation that follows; nothing else needs to hold a second copy of a
3-layer model on the GPU.

In [ ]:
if not have(UQMODEL, "(UQ artifacts)"):
    t0 = time.time()
    run(f"grace_uq build --model-yaml model.yaml --checkpoint checkpoints/checkpoint.best_test_loss.index "
        f"--train-data training_set.pkl.gz --rp-dim {RP_DIM} --n-clusters {N_CLUSTERS} "
        f"--threshold-percentile {UQ_PCTL} --n-workers 1 --restart "
        f"--artifact-path {UQART} --export-path {UQMODEL}", cwd=TSEED)
    print(f"\nUQ build + export took {time.time()-t0:.0f}s")

print(f"artifact     : {UQART.stat().st_size/1024:.0f} KB")
print(f"UQ SavedModel: {sum(f.stat().st_size for f in UQMODEL.rglob('*') if f.is_file())/1e6:.0f} MB")
run(f"grace_uq info {UQART} -v")

calc_uq = TPCalculator(str(UQMODEL))            # the teacher, with a gamma output
calc_uq.enable_uq(mode="gamma_only")            # per-atom gamma only; mode='full' adds the sigma fields
print("UQ modes available:", calc_uq.available_uq_modes)

## 6. Teacher validation — learning curves, parity, and the hull-only reference

In [ ]:
teacher_final = learning_curves(TSEED, "Teacher finetuning")
display(teacher_final)

In [ ]:
E, F = predict(test["ase_atoms"], calc_uq)                # calc_uq is the teacher (section 4)
teacher_mae = parity_plot(test["energy"].values, E, test["forces"], F, natoms(test),
                          ref="DFT", model=TEACHER_NAME,
                          title=f"Teacher on the {len(test)} held-out DFT structures")

# the teacher of the hull-only reference run on the same 16 structures (numbers shipped in 0-data/)
ref_summary = load(REF_SUMMARY)
display(pd.DataFrame({"hull + fps (this run)": teacher_mae,
                      "hull only (reference)": [ref_summary["teacher vs DFT, test set: E-MAE (meV/atom)"],
                                                ref_summary["teacher vs DFT, test set: F-MAE (meV/A)"]]},
                     index=["E-MAE (meV/atom)", "F-MAE (meV/A)"]).round(2))

## 7. UQ plots and the out-of-distribution demo

`LiAl_mp-1191737` sits **140 meV/atom above the convex hull** at $c_{Li}=0.5$ and was deliberately
held out of training. It is a *different prototype* at a composition the model otherwise knows well,
so a high γ here shows that γ responds to **structural** novelty, not merely to unseen compositions.

γ comes from the same ASE calculator as energies and forces (`TPCalculator` on the exported UQ
model, with `enable_uq`), so it is available in any Python workflow and not only through
`grace_uq predict`. Whether a high γ actually *predicts* a large error is tested in §8.

In [ ]:
def gamma_per_atom(atoms, calc):
    """One array of per-atom gamma per structure."""
    out = predict_structures(list(atoms), calc, properties=("energy",), extra=("gamma",))
    return [np.asarray(g, float).ravel() for g in out["gamma"]]

def teacher_gamma(atoms, tag, per="atom"):
    """Teacher gamma, cached under 2-uq-validation/. per='atom': every atom of every structure
    in one array; per='structure': the maximum over each structure."""
    f = UQDIR / (f"gamma_{tag}.npy" if per == "atom" else f"gamma_struct_{tag}.npy")
    if have(f): return np.load(f)
    t0 = time.time()
    g = gamma_per_atom(atoms, calc_uq)
    g = np.concatenate(g) if per == "atom" else np.array([x.max() for x in g])
    UQDIR.mkdir(exist_ok=True); np.save(f, g)
    print(f"   {tag}: {len(atoms)} structures in {time.time()-t0:.0f}s")
    return g

def gamma_stats(sets, thresholds=(1,), per="atom"):
    """Median, 90th percentile, maximum and the fraction above each threshold; one row per set.
    `per` says what each gamma value belongs to, 'atom' or 'structure', and appears in the headers."""
    rows = []
    for lab, g, *_ in sets:
        row = {f"n {per}s": len(g), "median": np.median(g), "p90": np.percentile(g, 90), "max": g.max()}
        row.update({f"% {per}s > {t:g}": 100*(g > t).mean() for t in thresholds})
        rows.append(pd.Series(row, name=lab))
    return pd.DataFrame(rows).round(2)

def gamma_hist(ax, sets, bins, xlabel):
    """Histogram of gamma per set as the fraction per log-spaced bin.
    (Weights rather than density=True: with log bins the bin width grows ~1000x across the axis.)"""
    for lab, g, color in sets:
        ax.hist(g, bins=bins, alpha=.55, color=color, weights=np.ones(len(g))/len(g),
                label=f"{lab}  (median {np.median(g):.2f}, {100*(g > 1).mean():.0f}% > 1)")
    ax.axvline(1.0, color="k", ls="--", lw=1.2, label=r"$\gamma = 1$")
    ax.set_xscale("log"); ax.set_xlabel(xlabel); ax.set_ylabel("fraction per bin"); ax.legend()

g_train = teacher_gamma(train_fps["ase_atoms"], "train")
g_test  = teacher_gamma(test["ase_atoms"],  "test")
g_ood   = teacher_gamma(ood["ase_atoms"],   "ood")
GAMMA_SETS = [("train", g_train, "tab:blue"), ("test (held out)", g_test, "tab:green"), ("OOD mp-1191737", g_ood, "tab:red")]

display(gamma_stats(GAMMA_SETS))
print(f"separation (OOD median / train median): {np.median(g_ood)/np.median(g_train):.0f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
gamma_hist(ax, GAMMA_SETS, np.logspace(-1.5, 3, 70), r"per-atom $\gamma$ (teacher, GMM)")
ax.set_yscale("log"); ax.set_title("Teacher uncertainty: in-distribution vs deliberately held-out structure")
plt.tight_layout(); plt.show()

## 8. What the finetuning bought — the convex hull, before and after

The parity plot is a per-structure view. The physically meaningful test is the **convex hull**, the
quantity that decides which phases are stable. So re-evaluate the whole dataset with the finetuned
teacher, rebuild the hull from its own energies exactly as in §2, and put the three hulls side by
side. Every structure is coloured by the teacher's per-structure γ (white at the calibrated threshold
γ = 1), so the one prototype it was never shown should stand out. The foundation-model hull is drawn
in grey; the finetuned hull is the dash-dot line whose vertices, DFT structures themselves, carry
their own γ colour.

In [ ]:
ft_file = TEACHER / "hull_pred.pkl.gz"
if not have(ft_file, "(teacher energies)"):
    t0 = time.time()
    E, _ = predict(ds["ase_atoms"], calc_uq)
    save(pd.DataFrame({"name": ds["name"], "E_ft": E}), ft_file)
    print(f"evaluated {len(ds)} structures in {time.time()-t0:.0f}s")
ft = load(ft_file)
assert (ft["name"].values == ds["name"].values).all(), "prediction rows out of order"

ds_ft   = with_hull(ds.assign(energy=ft["E_ft"].values))
hull_ft = lower_hull(ds_ft[fitted])

BEFORE, AFTER = "GRACE-3L-OMAT-large (before)", f"{TEACHER_NAME} (after)"
display(hull_table({BEFORE: hull_fm, AFTER: hull_ft}))
for lab, d in [(BEFORE, ds_fm), (AFTER, ds_ft)]:
    mae, worst = formation_mae(d)
    above = d.loc[~fitted, "e_chull_dist_per_atom"]*1000
    print(f"{lab:31s} formation-energy MAE {mae:4.1f} meV/atom (max {worst:4.1f}) | "
          f"OOD sits {above.min():.0f}-{above.max():.0f} meV/atom above its hull")

# per-structure extrapolation grade of the teacher for the whole dataset, OOD included
gam = teacher_gamma(ds["ase_atoms"], "dataset", per="structure")

from matplotlib.colors import TwoSlopeNorm
lg   = np.log10(np.clip(gam, 1e-3, None))                       # colour by log10(gamma), white at gamma = 1
norm = TwoSlopeNorm(vcenter=0.0, vmin=min(lg.min(), -0.05), vmax=max(lg.max(), 0.05))
# the vertices of the finetuned hull are DFT structures, so each carries its own gamma colour
ft_vertices = [ds_ft[fitted & (ds_ft["c_Li"].round(3) == round(c, 3))]["e_formation_per_atom"].idxmin() for c, _ in hull_ft]

fig, ax = plt.subplots(1, 2, figsize=(13.5, 5))
for a, xlim, ylim, ttl in [(ax[0], (-.02, 1.02), (-200, 60),   "full hull"),
                           (ax[1], (0.55, 0.75), (-192, -140), "zoom: the Li-rich end")]:
    for split, mk, sz in [("train", "o", 26), ("test", "o", 70), ("OOD", "^", 130)]:
        m = (ds["split"] == split).values
        a.scatter(ds.loc[m, "c_Li"], ds.loc[m, "e_formation_per_atom"]*1000, c=lg[m], norm=norm, cmap="bwr",
                  marker=mk, s=sz, zorder=2, edgecolor="0.25", linewidth=.4, label=f"{split} ({m.sum()})")
    plot_hull(a, hull,    "k.-", lw=2.2, ms=8, zorder=3)                              # DFT reference
    plot_hull(a, hull_fm, "s--", color="0.6", lw=1.8, ms=6, zorder=3)                 # before: grey
    plot_hull(a, hull_ft, "-.",  color="0.2", lw=1.6, zorder=3)                       # after: neutral line, vertices below
    a.scatter(ds_ft.loc[ft_vertices, "c_Li"], ds_ft.loc[ft_vertices, "e_formation_per_atom"]*1000, c=lg[ft_vertices],
              norm=norm, cmap="bwr", marker="D", s=85, zorder=4, edgecolor="k", linewidth=.8)
    a.set_xlim(*xlim); a.set_ylim(*ylim); a.set_title(ttl)
    a.set_xlabel("$c_{Li}$"); a.set_ylabel("formation energy (meV/atom)")
from matplotlib.lines import Line2D
handles, labels = ax[0].get_legend_handles_labels()
handles += [Line2D([], [], color="k",   ls="-",  marker="o", ms=8, lw=2.2, label="DFT (reference)"),
            Line2D([], [], color="0.6", ls="--", marker="s", ms=6, lw=1.8, label=BEFORE),
            Line2D([], [], color="0.2", ls="-.", marker="D", ms=7, mfc="w", mec="k", lw=1.6,
                   label=AFTER + r", vertices coloured by their $\gamma$")]
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, frameon=False)   # one legend below both panels
fig.subplots_adjust(bottom=0.24)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="bwr"), ax=ax, pad=0.015)
ticks = [t for t in (0.01, 0.1, 0.3, 1, 3, 10, 100, 1000) if norm.vmin <= np.log10(t) <= norm.vmax]
cb.set_ticks(np.log10(ticks)); cb.set_ticklabels([f"{t:g}" for t in ticks])
cb.set_label(r"teacher $\gamma$ (per-structure max), white = 1")
fig.suptitle(f"Al-Li convex hull before and after {N_UPDATES} frozen-readout updates, "
             r"symbols coloured by extrapolation grade $\gamma$")
plt.show()

print(f"gamma: train/test median {np.median(gam[fitted]):.2f} ({100*(gam[fitted] > 1).mean():.0f}% above 1); "
      f"OOD prototype median {np.median(gam[~fitted]):.1f} ({100*(gam[~fitted] > 1).mean():.0f}% above 1)")

### Does γ actually predict where the model is wrong?

The hull above was evaluated on fixed DFT geometries. In practice a potential *relaxes* candidate
structures, so here 19 Al–Li structures from Materials Project (shipped in `0-data/`) are relaxed
with the teacher, every relaxed candidate is scored with the teacher's γ, and γ is compared with the
candidate's error relative to the DFT hull. This is entirely a **teacher** measurement with the
**GMM γ** from its UQ artifact (§5 and §7). The student and its D-optimality γ play no part here; the
same relaxations are reused in §11, where the student's hull joins the comparison.

The reference matters. Measuring the same structures against Materials Project instead gives no
correlation at all; the offset between two DFT setups swamps the model error. **γ tracks error only
when error is measured against the reference the model was trained on.**

In [ ]:
from amstools.thermodynamics import run_convex_hull_calculation

HULLDIR.mkdir(exist_ok=True)
mp_candidates = load(MP_SET)

def relax_candidates(calc, tag):
    """Relax the Materials Project candidates with `calc`; cached under 4-convex-hull/."""
    f = HULLDIR / f"relaxed_{tag}.pkl.gz"
    if not have(f, f"({tag} relaxations)"):
        df, _ = run_convex_hull_calculation(structure_dict=dict(zip(mp_candidates["name"], mp_candidates["ase_atoms"])),
                                            calc=calc, pipeline_dict={}, verbose=False)
        df["ase_atoms"] = [a.copy() for a in df["ase_atoms"]]     # detach the calculators before saving
        save(df, f)
    return load(f)

teacher_relaxed = relax_candidates(calc_uq, "teacher")
teacher_relaxed["gamma"] = [g.max() for g in gamma_per_atom(teacher_relaxed["ase_atoms"], calc_uq)]
teacher_relaxed["err"]   = np.abs(teacher_relaxed["e_formation_per_atom"].astype(float)
                                  - hull_at(hull, teacher_relaxed["c_Li"].astype(float)))*1000

display(teacher_relaxed.groupby(np.where(teacher_relaxed["gamma"] > 10, "gamma > 10", "gamma <= 10"))["err"]
                       .agg(["count", "median"]).rename(columns={"median": "median |error vs DFT hull| (meV/atom)"}).round(1))

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(teacher_relaxed["gamma"], teacher_relaxed["err"], s=45)
for _, r in teacher_relaxed.nlargest(3, "gamma").iterrows():
    ax.annotate(str(r["name"]).split("__")[0], (r["gamma"], r["err"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xscale("log"); ax.axvline(1, color="k", ls="--", lw=1)
ax.set_xlabel(r"$\gamma$ (teacher)"); ax.set_ylabel("|error vs DFT hull| (meV/atom)")
ax.set_title(f"{TEACHER_NAME}: high uncertainty marks the relaxed structures it gets wrong")
plt.tight_layout(); plt.show()

## 9. Distillation pool

Plain ASE, no external structure database. Each hull structure is grown to a reasonable supercell,
then deformed: an isotropic volume scan plus random Voigt strains with rattling at several
amplitudes.

The pool is seeded from the **hull structures only** and generated exactly as in the hull-only
reference run, so the student recipe is identical and every difference in the student comes from the
teacher. The rattle amplitudes matter: the pool reaches |F|max of several eV/Å, and that force
diversity is what makes the student usable in MD.

In [ ]:
RNG = np.random.default_rng(42)
NAT_MIN, NAT_MAX = 16, 64            # supercell size window
VOL_RANGE, STRAIN = 0.08, 0.05       # +-8% volume scan, up to 5% random strain components
RATTLES = [0.03, 0.06, 0.10, 0.15]   # rattle amplitudes (A), cycled over the random samples

def grow(at):
    """Repeat the shortest cell vector until the cell has at least NAT_MIN atoms."""
    at = at.copy()
    while len(at) < NAT_MIN and len(at)*2 <= NAT_MAX:
        rep = [1, 1, 1]; rep[int(np.argmin(at.cell.lengths()))] = 2
        at = at*tuple(rep)
    return at

def voigt(eps):
    """Random symmetric deformation matrix with components in [-eps, eps]."""
    e = RNG.uniform(-eps, eps, 6)
    return np.array([[1+e[0], e[5]/2, e[4]/2],
                     [e[5]/2, 1+e[1], e[3]/2],
                     [e[4]/2, e[3]/2, 1+e[2]]])

def deform(at0):
    """Volume scan plus strained-and-rattled copies of one seed structure."""
    out, base = [], grow(at0)
    for f in np.linspace(1-VOL_RANGE, 1+VOL_RANGE, POOL_NVOL):
        a = base.copy(); a.set_cell(base.cell.array*f**(1/3), scale_atoms=True); out.append(a)
    for i in range(POOL_NRAND):
        a = base.copy()
        a.set_cell(base.cell.array @ voigt(STRAIN), scale_atoms=True)
        a.positions += RNG.normal(0, RATTLES[i % len(RATTLES)], a.positions.shape)
        out.append(a)
    return out

DISTILL.mkdir(exist_ok=True)
pool_file = DISTILL / "pool_unlabelled.pkl.gz"
if not have(pool_file, "(pool structures)"):
    seeds = train[natoms(train) <= 13]                       # primitive cells only
    rows  = [{"name": f"pool_{i:03d}_{j:02d}", "ase_atoms": a}
             for i, at in enumerate(seeds["ase_atoms"]) for j, a in enumerate(deform(at))]
    save(pd.DataFrame(rows), pool_file)
    print(f"generated from {len(seeds)} seeds")
pool_structures = load(pool_file)
print(f"pool: {len(pool_structures)} structures, {natoms(pool_structures).sum()} atoms")

### Label the pool with the teacher

`grace_predict` evaluates a whole dataframe in one go. It writes energies and forces keyed by
`name`; merging them back onto the pool structures gives the student's training set.

In [ ]:
distilled_file = DISTILL / "distilled.pkl.gz"
if not have(distilled_file, "(teacher labels)"):
    labels_file = DISTILL / "pool_pred.pkl.gz"
    t0 = time.time()
    run(f"grace_predict --model {TSEED}/final_model --dataset {pool_file} --output {labels_file}")
    labels = load(labels_file).rename(columns={"energy_predicted": "energy", "forces_predicted": "forces"})
    save(pool_structures.merge(labels, on="name")[["name", "ase_atoms", "energy", "forces"]], distilled_file)
    print(f"\nlabelling took {time.time()-t0:.0f}s")

pool = load(distilled_file)                                  # the pool with the teacher's labels
pool["nat"], pool["fmax"] = natoms(pool), fmax(pool)
pool["epa"] = pool["energy"]/pool["nat"]
print(f"distilled pool: {len(pool)} structures, {pool['nat'].sum()} atoms")
print(f"  E/atom : {pool['epa'].min():.3f} .. {pool['epa'].max():.3f} eV")
print(f"  |F|max : median {pool['fmax'].median():.3f}, max {pool['fmax'].max():.2f} eV/A  "
      f"(teacher training set: median {np.median(fmax(train_fps)):.3f})")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(pool["epa"], bins=60); ax[0].set_xlabel("E/atom (eV)"); ax[0].set_ylabel("count")
ax[1].hist(pool["fmax"], bins=60); ax[1].set_xlabel("|F|max per structure (eV/A)"); ax[1].set_yscale("log")
fig.suptitle("Distillation pool"); plt.tight_layout(); plt.show()

### How reliable are the teacher's labels?

The pool was built by rattling and straining the hull structures, so some of it necessarily falls
outside what the teacher actually knows. Those labels are the teacher's *extrapolation*, not its
knowledge, and the student will learn them just as faithfully as the good ones.

The teacher's own γ answers this directly: score every pool structure with the UQ model from §5
and compare against the γ of the teacher's training set. This is also why the student is compared
against the *teacher* in §10: it cannot be better than its labels.

**Read the table per structure.** Each value here is the *largest* γ among a structure's atoms, the
number that decides whether a whole structure can be trusted. The calibration put γ = 1 at the 99th
percentile of the training *atoms*, so about 1 % of them lie above it; a structure is flagged as
soon as one of its atoms is, so with cells of 6 to 100 atoms roughly 10 to 15 % of the training
*structures* show a maximum above 1. That is the same 1 % of atoms seen through a coarser lens, not
a failure of the calibration. The per-atom view is the table in §7. In this tutorial expect the flagged
training structures to be fps-selected cells: the threshold lands where the training data is sparse,
and that is exactly where the selection added structures.

In [ ]:
g_pool     = teacher_gamma(pool["ase_atoms"],  "pool",  per="structure")
g_train_st = teacher_gamma(train_fps["ase_atoms"], "train", per="structure")
POOL_SETS  = [("teacher training set", g_train_st, "tab:blue"), ("distilled pool", g_pool, "tab:purple")]
display(gamma_stats(POOL_SETS, thresholds=(1, 10), per="structure"))     # max gamma over each structure's atoms

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
lo, hi = min(g_pool.min(), g_train_st.min()), max(g_pool.max(), g_train_st.max())
gamma_hist(ax[0], POOL_SETS, np.logspace(np.log10(max(lo, 1e-2)), np.log10(hi), 55), r"per-structure max $\gamma$ (teacher)")
ax[0].set_title("Is the pool inside what the teacher knows?")
ax[1].scatter(pool["fmax"], g_pool, s=6, alpha=.25)
ax[1].axhline(1, color="k", ls="--", lw=1.2); ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("|F|max per structure (eV/A)"); ax[1].set_ylabel(r"max $\gamma$")
ax[1].set_title("Uncertainty grows with how hard the structure was pushed")
plt.tight_layout(); plt.show()
print(f"{100*(g_pool > 1).mean():.0f}% of the pool structures contain at least one atom with gamma > 1: "
      f"somewhere inside them the teacher is extrapolating")

### Keep only the labels the teacher stands behind?

If a pool structure is extrapolative for the teacher, its label is a guess. The obvious response is
to keep only the structures whose largest atomic γ is at most 1 and distil from those.

**Do this with your eyes open.** γ rises with how hard a structure was pushed (right panel above),
so a γ cut is very nearly a force cut. A model fit only to near-equilibrium structures has never
been shown a restoring force of any size and will be soft in MD. The cell below shows what the cut
removes (γ ≤ 1 when no cut is applied). `POOL_GAMMA_MAX = None` in §0 trains the student on the whole
pool; a number trains it on the structures whose largest γ is below it. The setting is part of the
student's directory name, so changing it trains a new student instead of reusing the old one.

**Why the cut can sit well above γ = 1.** The threshold marks where the teacher leaves the data it
was *finetuned* on, not where its knowledge ends. The teacher is a strong foundation model finetuned
with frozen weights: only the readout was trained, everything else still carries what
`GRACE-3L-OMAT-large` learned from OMat24, so there is little catastrophic forgetting and its labels
stay reliable well beyond the calibrated threshold. That is why the default here is
`POOL_GAMMA_MAX = 5` rather than 1: it removes only the far tail of the pool and keeps the force
diversity the student needs.

The structures a cut drops are not waste: they are precisely the ones worth computing in DFT, which
is the same selection `pace_select` performs on the MD trajectory in §13. Active learning, applied
at the distillation stage instead of after the run.

In [ ]:
cut    = 1.0 if POOL_GAMMA_MAX is None else POOL_GAMMA_MAX
keep   = g_pool <= cut
groups = {"full pool": np.ones(len(pool), bool), f"max gamma <= {cut:g}": keep}
display(pd.DataFrame({lab: {"structures": m.sum(), "atoms": pool.loc[m, "nat"].sum(),
                            "|F|max median": np.median(pool.loc[m, "fmax"]),
                            "|F|max p90": np.percentile(pool.loc[m, "fmax"], 90),
                            "|F|max max": pool.loc[m, "fmax"].max()} for lab, m in groups.items()}).T.round(3))
print(f"keeping only structures with max gamma <= {cut:g} keeps {100*keep.mean():.0f}% of them and lowers the |F|max ceiling "
      f"from {pool['fmax'].max():.2f} to {pool.loc[keep, 'fmax'].max():.2f} eV/A")

if POOL_GAMMA_MAX is None:
    student_data = distilled_file
    print("POOL_GAMMA_MAX is None: the student trains on the full pool")
else:
    student_data = DISTILL / f"distilled-{POOL_TAG}.pkl.gz"
    save(pool.loc[keep, ["name", "ase_atoms", "energy", "forces"]].reset_index(drop=True), student_data)
    print(f"the student trains on the {keep.sum()} trusted structures in {student_data.name}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.logspace(np.log10(max(pool["fmax"].min(), 1e-4)), np.log10(pool["fmax"].max()), 45)
for (lab, m), color in zip(groups.items(), ["tab:purple", "tab:green"]):
    f = pool.loc[m, "fmax"]
    ax[0].hist(f, bins=bins, alpha=.6, color=color, weights=np.ones(len(f))/len(f), label=lab)
ax[0].set_xscale("log"); ax[0].set_xlabel("|F|max per structure (eV/A)"); ax[0].set_ylabel("fraction of structures per bin")
ax[0].legend(); ax[0].set_title("What a gamma cut removes: the forces")
ax[1].scatter(pool.loc[~keep, "nat"], pool.loc[~keep, "fmax"], s=7, alpha=.25, color="tab:red",   label=f"dropped ({(~keep).sum()})")
ax[1].scatter(pool.loc[keep,  "nat"], pool.loc[keep,  "fmax"], s=7, alpha=.35, color="tab:green", label=f"kept ({keep.sum()})")
ax[1].set_yscale("log"); ax[1].set_xlabel("atoms per structure"); ax[1].set_ylabel("|F|max (eV/A)")
ax[1].legend(); ax[1].set_title("Kept vs dropped")
plt.tight_layout(); plt.show()

### Does the distilled pool still describe the convex hull?

The student will only ever see these teacher-generated labels, so anything wrong here is baked into
it. Two things to check: that the pool still reproduces the DFT hull, and that it extends well
*above* the hull. That spread is what gives the student the force diversity it needs for MD.

In [ ]:
pool_hull = with_hull(pool)
hull_pool = lower_hull(pool_hull)
display(hull_table({"distilled pool": hull_pool}))
ef, above = pool_hull["e_formation_per_atom"]*1000, pool_hull["e_chull_dist_per_atom"]*1000
print(f"pool spans {ef.min():.0f} .. {ef.max():.0f} meV/atom ({100*(above > 100).mean():.0f}% of it more than 100 meV/atom above its hull)")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, ylim, ttl in [(ax[0], (-210, 520), "full pool"), (ax[1], (-210, 20), "zoom on the hull")]:
    a.scatter(pool_hull["c_Li"], ef, s=7, alpha=.18, color="tab:purple", zorder=2, label=f"distilled ({len(pool_hull)})")
    plot_hull(a, hull,      "k.-", lw=1.6, ms=10, zorder=4, label="DFT hull")
    plot_hull(a, hull_pool, "--",  color="tab:purple", lw=1.6, zorder=3, label="distilled hull")
    a.set_xlabel("$c_{Li}$"); a.set_ylabel("formation energy (meV/atom)")
    a.set_ylim(*ylim); a.set_title(ttl); a.legend(loc="upper center", fontsize=9)
fig.suptitle("Convex hull of the distilled data vs the DFT reference")
plt.tight_layout(); plt.show()

## 10. Distil into a GRACE/FS student

**Finetune `GRACE-FS-OMAT` rather than fitting FS from scratch.** Same cost per BFGS iteration,
but roughly 3× fewer iterations for a better model: finetuning at 100 iterations beats
from-scratch at 250 on both energy and forces.

GRACE/FS has no frozen-readout mode, so all of its parameters are trained.

### Before finetuning — the hull straight out of `GRACE-FS-OMAT`

The student starts from a foundation model of its own. `GRACE-FS-OMAT` was trained on the same
OMat24 data as the teacher, but it is a linear model and far cheaper to evaluate. Before distilling
anything, give it the treatment of §2: evaluate the dataset, rebuild the hull from its own energies,
and compare the shape with DFT and with the `GRACE-3L-OMAT-large` hull. This is the starting point
the distillation has to improve on; the section on relaxed structures below puts the student's hull
after finetuning next to it.

In [ ]:
fsomat_file = FMDIR / "fsomat_pred.pkl.gz"
calc_fsomat = grace_fm("GRACE-FS-OMAT")                # the student before finetuning; kept for the relaxed hulls below
if not have(fsomat_file, "(GRACE-FS-OMAT energies)"):
    t0 = time.time()
    E, _ = predict(ds["ase_atoms"], calc_fsomat)
    save(pd.DataFrame({"name": ds["name"], "E_fsomat": E}), fsomat_file)
    print(f"evaluated {len(ds)} structures in {time.time()-t0:.0f}s")
fsomat = load(fsomat_file)
assert (fsomat["name"].values == ds["name"].values).all(), "prediction rows out of order"

ds_fsomat   = with_hull(ds.assign(energy=fsomat["E_fsomat"].values))   # hull from GRACE-FS-OMAT's OWN energies
hull_fsomat = lower_hull(ds_fsomat[fitted])

display(hull_table({"GRACE-3L-OMAT-large": hull_fm, "GRACE-FS-OMAT": hull_fsomat}))
for lab, d in [("GRACE-3L-OMAT-large", ds_fm), ("GRACE-FS-OMAT", ds_fsomat)]:
    mae, worst = formation_mae(d)
    print(f"{lab:20s} formation-energy MAE vs DFT {mae:4.1f} meV/atom (max {worst:4.1f})")
print(f"hull minimum: DFT {min(e for _, e in hull)*1000:.1f} vs GRACE-FS-OMAT {min(e for _, e in hull_fsomat)*1000:.1f} meV/atom")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.scatter(ds.loc[fitted, "c_Li"], ds.loc[fitted, "e_formation_per_atom"]*1000, s=18, color="0.8", zorder=1, label="DFT structures")
plot_hull(ax, hull,        "k.-", lw=2.2, ms=9, zorder=3, label="DFT hull (reference)")
plot_hull(ax, hull_fm,     "s--", color="0.6", lw=1.6, ms=5, zorder=2, label="GRACE-3L-OMAT-large (no finetuning)")
plot_hull(ax, hull_fsomat, "^--", color="tab:green", lw=1.8, ms=6, zorder=2, label="GRACE-FS-OMAT (no finetuning)")
ax.set_xlabel("$c_{Li}$"); ax.set_ylabel("formation energy (meV/atom)")
ax.set_title("Al-Li convex hull straight out of the two foundation models")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
STUDENT.mkdir(parents=True, exist_ok=True)
(STUDENT / "input.yaml").write_text(Template("""
seed: 1
cutoff: n/a

data:
  filename: ../$DATA_FILE
  test_size: 0.05
  reference_energy: 0
  save_dataset: True

potential:
  finetune_foundation_model: GRACE-FS-OMAT
  reduce_elements: True
  shift: auto

fit:
  # no stress term: the distilled labels carry energy and forces only
  loss:
    energy: {weight: 16, type: huber, delta: 0.01}
    forces: {weight: 32, type: huber, delta: 0.01}
  maxiter: $FS_ITERS
  optimizer: BFGS
  opt_params: {maxcor: 100, maxls: 20, gtol: 1.e-8, iprint: -1}
  compute_convex_hull: False
  batch_size: 16
  test_batch_size: 64
  jit_compile: True
  eval_init_stats: True
  train_max_n_buckets: 2
  test_max_n_buckets: 1
  checkpoint_freq: 20
  progressbar: False
""").substitute(DATA_FILE=student_data.name, FS_ITERS=FS_ITERS))
print("written", STUDENT / "input.yaml", "- training on", student_data.name)

In [ ]:
if not have(SSEED / "final_model", "(student already trained)"):
    t0 = time.time()
    run("gracemaker input.yaml", cwd=STUDENT)
    print(f"\nstudent finetune took {time.time()-t0:.0f}s")

### Build the active set — `grace_utils export -sf` + `pace_activeset`

Everything that follows (the speed comparison, the student's extrapolation grade, the hull and the
LAMMPS run) needs the student in its exported form: `saved_model.yaml`, read by the C++ evaluator,
and its active set `saved_model.asi`. The active set is what turns a plain potential into one that
can report when it is extrapolating.

In [ ]:
fs_yaml, fs_asi = SSEED / "saved_model.yaml", SSEED / "saved_model.asi"
if not have(fs_yaml, "(FS export)"):
    run("grace_utils -p model.yaml -c checkpoints/checkpoint.best_test_loss.index export -sf", cwd=SSEED)
if not have(fs_asi, "(active set)"):
    run("pace_activeset -d training_set.pkl.gz saved_model.yaml", cwd=SSEED)
print(f"{fs_yaml.name}: {fs_yaml.stat().st_size/1024:.0f} KB | {fs_asi.name}: {fs_asi.stat().st_size/1024:.0f} KB")

### Student validation — learning curves and parity

The same two views as §6, now for the student, but note what the reference is.

The teacher was scored against **DFT**. The student is scored against **the teacher's own labels**,
because that is all it was ever shown. These plots therefore measure *distillation fidelity*, how
faithfully the small model reproduces the big one, not accuracy against first principles. So the
cell after the parity plot also scores the student on the teacher's held-out DFT structures. Its
error against DFT is what it inherits from the teacher plus what it loses in distillation; the two
contributions are measured on different sets, so they bound rather than exactly sum to the third row.

BFGS iterations replace epochs on the x-axis: the student is fit with a full-batch quasi-Newton
optimizer, not Adam.

In [ ]:
student_final = learning_curves(SSEED, f"Student finetuning - GRACE-FS-OMAT, {FS_ITERS} BFGS iterations", xlabel="BFGS iteration")
display(pd.concat({"teacher (vs DFT)": teacher_final, "student (vs teacher labels)": student_final}, axis=1))

In [ ]:
calc_s   = TPCalculator(str(SSEED / "final_model"))
stu_test = load(SSEED / "test_set.pkl.gz")                 # gracemaker's 5% split of the pool, teacher labels
E, F = predict(stu_test["ase_atoms"], calc_s)
student_vs_teacher = parity_plot(stu_test["energy"].values, E, stu_test["forces"], F, natoms(stu_test),
                                 ref="teacher", model="student", color="tab:green",
                                 title=f"Student vs teacher on {len(stu_test)} held-out distilled structures")

# the same student on the teacher's held-out DFT structures
E, F = predict(test["ase_atoms"], calc_s)
student_vs_dft = errors(test["energy"].values, E, test["forces"], F, natoms(test))

display(pd.DataFrame([teacher_mae, student_vs_teacher, student_vs_dft],
                     index=["teacher vs DFT", "student vs teacher", "student vs DFT"],
                     columns=["E-MAE (meV/atom)", "F-MAE (meV/A)"])
          .assign(reference=[f"DFT, {len(test)} held-out structures",
                             f"teacher labels, {len(stu_test)} held-out pool structures",
                             f"DFT, the same {len(test)} structures"]).round(2))

### The point of distillation: speed

`PyGRACEFSCalculator` runs the C++ ACE evaluator, the same code path LAMMPS uses, driven by the
exported `saved_model.yaml` and its active set. Timing it against the 3-layer teacher on the same
structures is the whole argument for distilling. The 16 small test cells understate the gap; it
grows with system size.

In [ ]:
from pyace.asecalc import PyGRACEFSCalculator

calc_fs = PyGRACEFSCalculator(str(fs_yaml))      # the student exactly as LAMMPS will use it
calc_fs.set_active_set(str(fs_asi))

def timed(calc, atoms):
    """Wall-clock for one sequential pass over the structures, in the same order for both calculators.
    (Sorting by size would let the TF model amortise its compilations and narrow the gap.)"""
    t0 = time.time()
    E = predict_structures(atoms, calc, properties=("energy",), sort_by_natoms=False)["energy"]
    return np.array(E)/np.array([len(a) for a in atoms]), time.time() - t0

probe = test["ase_atoms"].tolist()
E_fs, t_fs = timed(calc_fs, probe)
calc_uq.disable_uq()                             # plain energies, for a like-for-like timing
E_tf, t_tf = timed(calc_uq, probe)
calc_uq.enable_uq(mode="gamma_only")
print(f"teacher  GRACE-3L (TensorFlow, GPU): {t_tf:6.2f} s")
print(f"student  GRACE-FS (C++, 1 CPU core): {t_fs:6.2f} s   -> {t_tf/t_fs:.0f}x faster")
print(f"mean |E/atom| difference student vs teacher: {np.abs(E_fs - E_tf).mean()*1000:.2f} meV/atom")

### Student uncertainty on the convex hull

The student's γ is the D-optimality extrapolation grade from `pace_activeset`, the same number
LAMMPS prints during MD. It is a **different quantity** from the teacher's GMM γ of §5 and §7–8: the two
rank structures the same way, but the D-optimality γ has a much smaller dynamic range. Never compare
the two scales.

In [ ]:
def student_gamma(atoms):
    """Per-atom D-optimality gamma of the student, all atoms in one array."""
    return np.concatenate(gamma_per_atom(atoms, calc_fs))

stu_train = load(SSEED / "training_set.pkl.gz")
on_hull   = ds[fitted & (ds["e_chull_dist_per_atom"]*1000 < 1.0)]
STUDENT_SETS = [("student training pool (200 sampled)", student_gamma(stu_train.sample(200, random_state=0)["ase_atoms"])),
                (f"dataset on the hull (n={len(on_hull)})", student_gamma(on_hull["ase_atoms"])),
                ("OOD mp-1191737",                         student_gamma(ood["ase_atoms"]))]
display(gamma_stats(STUDENT_SETS, thresholds=(1, 1.5)))

### Energy and force errors against DFT — all three models

Everything so far was measured either on the teacher's own 16-structure test set or on the hull.
This is the broader check: **`AlLi-extended-eval.pkl.gz`**, 265 held-out DFT structures prepared
offline from the same source database (`0-data/make_extended_eval.py`). It is deliberately wider
than train+test: 16 compositions instead of 7, and deformation types the training set never
contained (`elast`, `shake0`–`shake4`, `def`, plus generated rattled cells), with the train, test
and OOD structures removed, `nn` scans dropped, and everything beyond 500 meV/atom above the hull
or with atoms closer than 2 Å or further apart than the 6 Å cutoff filtered out.

**On the energy column.** The foundation model was trained against a different DFT setup, so its
absolute energies carry a constant offset against this reference; a raw MAE would measure that
offset, not the model. The table therefore reports the MAE after removing each model's own mean
signed error, and prints the shift alongside so it is visible. For the two finetuned models the
shift is small by construction. Forces need no such correction.

The rows of the **hull-only reference run** are shown alongside, from the reference file shipped in
`0-data/`; the foundation-model row is the same model in both and appears once.

In [ ]:
def errors_vs_dft(df, calc):
    """Energy MAE with the model's constant offset removed, the offset itself, and the force MAE."""
    E, F = predict(df["ase_atoms"], calc)
    d = (E - df["energy"].values)/natoms(df)*1000
    return {"shift (meV/at)": d.mean(), "E-MAE (meV/at)": np.abs(d - d.mean()).mean(),
            "F-MAE (meV/A)": np.abs(flat(F) - flat(df["forces"])).mean()*1000}

errors_file = DISTILL / f"model_errors-{POOL_TAG}.pkl.gz"
if not have(errors_file, "(model errors)"):
    extended = load(EXT_SET)
    sets   = {f"test set ({len(test)})": test, f"extended DFT ({len(extended)})": extended}
    models = {"GRACE-3L-OMAT-large (foundation)": calc_fm,
              f"{TEACHER_NAME} (teacher)":         calc_uq,
              "GRACE-FS-distilled (student)":     calc_fs}
    rows, t0 = [], time.time()
    for sname, sdf in sets.items():
        for mname, calc in models.items():
            rows.append({"set": sname, "model": mname, **errors_vs_dft(sdf, calc)})
            print(f"   {mname:34s} on {sname:22s} done ({time.time()-t0:.0f}s)")
    save(pd.DataFrame(rows), errors_file)

model_errors = load(errors_file).assign(training="hull + fps (this run)")
reference    = load(REF_ERRORS)
reference    = reference[~reference["model"].str.contains("foundation")].assign(training="hull only (reference)")
both = pd.concat([model_errors, reference], ignore_index=True)
both["model"] = both["model"].str.replace(r"GRACE-3L-hull(\+fps)?-ft", "GRACE-3L finetuned", regex=True)
for col in ("E-MAE (meV/at)", "F-MAE (meV/A)", "shift (meV/at)"):
    print(f"\n{col}")
    display(both.pivot(index=["model", "training"], columns="set", values=col).round(2))

## 11. Convex hull from relaxed structures — DFT vs teacher vs student

The 19 Al–Li structures from Materials Project (shipped in `0-data/`)
were relaxed with the teacher in §8. Here the student relaxes them as well, and so does its starting
point `GRACE-FS-OMAT`, so the plot shows what the distillation bought. All hulls are compared with the
DFT hull of the tutorial dataset.

The reference is the tutorial's own DFT data, not Materials Project energies. MP and this dataset
were computed with different DFT settings and their hulls differ by tens of meV/atom (MP puts the
minimum at $c_{Li}=0.5$, this data at $c_{Li}=0.6$); comparing to MP would measure that difference,
not the model.

In [ ]:
from amstools.thermodynamics import plot_convex_hull

relaxed = {"GRACE-FS-OMAT (student before finetuning)": relax_candidates(calc_fsomat, "fsomat"),
           f"{TEACHER_NAME} (teacher)":                  teacher_relaxed,             # relaxed where gamma was compared with error
           "GRACE-FS-distilled (student)":               relax_candidates(calc_fs, f"student-{POOL_TAG}")}

# DFT reference: the lowest tutorial structure at each composition
lowest  = ds[fitted].groupby(ds.loc[fitted, "c_Li"].round(4))["energy_per_atom"].idxmin()
dft_ref = ds.loc[lowest].reset_index(drop=True)

plot_convex_hull({"DFT (tutorial dataset)": dft_ref, **relaxed}, figsize=(10, 6))
plt.ylim(-0.21, 0.06); plt.title("Al-Li convex hull vs the tutorial DFT reference"); plt.show()

minima = {}
for name, df in {"DFT (tutorial dataset)": dft_ref, **relaxed}.items():
    ef = df["e_formation_per_atom"].astype(float)
    minima[name] = {"hull minimum (meV/atom)": ef.min()*1000, "at c_Li": df.loc[ef.idxmin(), "c_Li"]}
display(pd.DataFrame(minima).T.round(3))

## 12. Validation beyond the hull — energy–volume curves, elastic constants, phonons

The hull tests energies at fixed geometries. A potential that is going to drive MD also has to get
the *shape* of the energy landscape right around each minimum: how the energy rises when the cell is
compressed or sheared, and how the atoms vibrate about their sites. `amstools` chains the standard
checks into one pipeline

```python
StepwiseOptimizer() + MurnaghanCalculator() + ElasticMatrixCalculator() + PhonopyCalculator()
```

relax the cell, scan the energy–volume curve and fit it (equilibrium volume and bulk modulus), apply
symmetry-adapted strains for the elastic constants, and displace atoms in a supercell for the phonons
through `phonopy`. The same pipeline runs for the four models of this notebook,

* the two foundation models untouched, `GRACE-3L-OMAT-large` and `GRACE-FS-OMAT`,
* the finetuned teacher and the distilled student,

on fcc Al and on the B32 AlLi compound at $c_{Li} = 0.5$. Neither structure is in the training data
as such; the training set covers them only through the strained and rattled cells of their
compositions. Every model is evaluated at *its own* relaxed cell, so the numbers are what each
potential would give in a simulation. The 3-layer models take a few minutes per structure on the GPU,
the FS models seconds; the results are cached.

In [ ]:
from ase.build import bulk
from ase import Atoms
from amstools import StepwiseOptimizer, MurnaghanCalculator, ElasticMatrixCalculator, PhonopyCalculator
from phonopy.phonon.band_structure import get_band_qpoints_and_path_connections

VALID = OUT / "6-validation"
VALID.mkdir(exist_ok=True)

# primitive cells of the fcc lattice for both, so the cubic lattice constant is a = (4 V_cell)^(1/3)
A_AL, A_ALLI = 4.05, 6.37
STRUCTURES = {"fcc Al":   bulk("Al", "fcc", a=A_AL),
              "B32 AlLi": Atoms("Al2Li2", cell=bulk("Al", "fcc", a=A_ALLI).cell, pbc=True,      # two interpenetrating
                                positions=np.array([[0, 0, 0], [.25, .25, .25],                  # diamond sublattices
                                                    [.5, .5, .5], [.75, .75, .75]])*A_ALLI)}
MODELS = {"GRACE-3L-OMAT-large":          (calc_fm,     "3L-OMAT"),          # calculator, cache tag
          TEACHER_NAME:                   (calc_uq,     "3L-ft"),
          "GRACE-FS-OMAT":                (calc_fsomat, "FS-OMAT"),
          "GRACE-FS-distilled (student)": (calc_fs,     f"FS-ft-{POOL_TAG}")}

# high-symmetry path of the fcc Brillouin zone, in reciprocal coordinates of the primitive cell
Q_PATH   = [[[0, 0, 0], [.5, 0, .5], [.5, .25, .75], [.375, .375, .75], [0, 0, 0], [.5, .5, .5]]]
Q_LABELS = ["Γ", "X", "W", "K", "Γ", "L"]

def validate(atoms, calc, cache):
    """Relax `atoms` with `calc`, then energy-volume curve, elastic constants and phonons; cached."""
    if have(cache):
        return load(cache)
    pipe = (StepwiseOptimizer(verbose=False) + MurnaghanCalculator(verbose=False)
            + ElasticMatrixCalculator(verbose=False) + PhonopyCalculator(verbose=False))
    pipe.run(init_structure=atoms, engine=calc)
    murn, C, phon = pipe["murnaghan"].value, np.array(pipe["elastic_matrix"].value["C"]), pipe["phonons"].phonopy
    qpts, connections = get_band_qpoints_and_path_connections(Q_PATH, npoints=51)
    phon.run_band_structure(qpts, path_connections=connections, labels=Q_LABELS)
    bands = phon.get_band_structure_dict()
    nat = len(atoms)
    res = pd.Series({"a0 (A)":              (4*murn["equilibrium_volume"])**(1/3),
                     "B (GPa)":             murn["equilibrium_bulk_modulus"],
                     "C11 (GPa)":           C[0, 0], "C12 (GPa)": C[0, 1], "C44 (GPa)": C[3, 3],
                     "lowest phonon (THz)": min(f.min() for f in bands["frequencies"]),   # negative = imaginary mode
                     "V (A^3/atom)":        np.array(murn["volume"])/nat,
                     "E-E0 (meV/atom)":     (np.array(murn["energy"]) - murn["equilibrium_energy"])/nat*1000,
                     "band distances":      bands["distances"], "band frequencies": bands["frequencies"]})
    save(res, cache)
    return res

results = {}
for sname, atoms in STRUCTURES.items():
    for mname, (calc, tag) in MODELS.items():
        t0 = time.time()
        results[(sname, mname)] = validate(atoms, calc, VALID / f"{sname.replace(' ', '-')}_{tag}.pkl.gz")
        print(f"{sname:9s} {mname:30s} {time.time()-t0:5.0f} s")

The table first. A negative *lowest phonon* frequency is an imaginary mode: the structure is
dynamically unstable for that model. Experimental room-temperature values for Al are listed for
orientation; DFT itself misses them by a few percent, so agreement to that level is all a
DFT-trained model can offer.

In [ ]:
COLS = ["a0 (A)", "B (GPa)", "C11 (GPa)", "C12 (GPa)", "C44 (GPa)", "lowest phonon (THz)"]
rows = [pd.Series(r[COLS], name=(s, m)) for (s, m), r in results.items()]
rows.append(pd.Series(dict(zip(COLS[:5], [4.05, 76, 107, 61, 28])), name=("fcc Al", "experiment (room temperature)")))
table = pd.DataFrame(rows).astype(float)
table.index = pd.MultiIndex.from_tuples(table.index, names=["structure", "model"])
display(table.sort_index(level=0, sort_remaining=False).round(2))

Then the curves: energy–volume around each model's own minimum, and the phonon dispersion along
Γ–X–W–K–Γ–L. Grey and dashed are the foundation models before finetuning; solid lines are the
finetuned teacher and the student it taught. Where the student's dispersion follows the teacher's,
the distillation has transferred not just energies but curvature; where a foundation model's curve
moves after finetuning, the 256 DFT structures changed the model's picture of this phase.

In [ ]:
from matplotlib.lines import Line2D

STYLE = {"GRACE-3L-OMAT-large":          dict(color="0.55",      ls="--"),
         TEACHER_NAME:                   dict(color="tab:blue",  ls="-"),
         "GRACE-FS-OMAT":                dict(color="tab:green", ls="--"),
         "GRACE-FS-distilled (student)": dict(color="tab:red",   ls="-")}

fig, ax = plt.subplots(2, 2, figsize=(13, 8.5))
for j, sname in enumerate(STRUCTURES):
    for mname in MODELS:
        r, st = results[(sname, mname)], STYLE[mname]
        ax[0, j].plot(r["V (A^3/atom)"], r["E-E0 (meV/atom)"], marker="o", ms=4, **st)
        total = r["band distances"][-1][-1]                                   # x in fractions of the whole path:
        for d, f in zip(r["band distances"], r["band frequencies"]):          # the cells differ slightly between models
            ax[1, j].plot(d/total, f, lw=1.1, **st)
    ticks = [d[0]/total for d in r["band distances"]] + [1.0]
    ax[1, j].set_xticks(ticks); ax[1, j].set_xticklabels(Q_LABELS); ax[1, j].set_xlim(0, 1)
    for t in ticks: ax[1, j].axvline(t, color="0.85", lw=.7, zorder=0)
    ax[1, j].axhline(0, color="k", lw=.6)
    ax[0, j].set_title(f"{sname}: energy-volume curve");  ax[0, j].set_xlabel("volume (A$^3$/atom)"); ax[0, j].set_ylabel("E - E$_0$ (meV/atom)")
    ax[1, j].set_title(f"{sname}: phonon dispersion");    ax[1, j].set_ylabel("frequency (THz)")
fig.legend(handles=[Line2D([], [], lw=2, label=m, **st) for m, st in STYLE.items()], loc="lower center", ncol=4, frameon=False)
fig.subplots_adjust(bottom=0.09); plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

## 13. LAMMPS MD with on-the-fly extrapolation grade

`pair_style grace/fs extrapolation` computes the student's per-atom γ during the run. Frames where
γ exceeds 1.5 are dumped; the run halts if γ exceeds 25.

The teacher was trained on 128 near-equilibrium structures, so a heating ramp **is expected** to
leave the training domain. That is the point: γ rises, candidate structures accumulate, and the run
stops itself rather than producing silent nonsense.

The run uses the Kokkos/CUDA build of LAMMPS on the T4 (`-sf kk` turns `grace/fs` into `grace/fs/kk`).
The results of the run are shipped with the repository; re-running needs the LAMMPS binary from the
setup cell.

In [ ]:
from ase.io import write as ase_write

MD.mkdir(exist_ok=True)
data_file = MD / "AlLi.lammps-data"
if not have(data_file, "(MD start structure)"):
    base = train[train["name"].astype(str).str.contains("LiAl_mp-1067/opt", regex=False)].iloc[0]["ase_atoms"]
    cell = base*(6, 6, 6)
    cell = cell[np.argsort([{"Al": 0, "Li": 1}[s] for s in cell.get_chemical_symbols()], kind="stable")]  # Al first, then Li
    ase_write(data_file, cell, format="lammps-data", specorder=["Al", "Li"], masses=True)
    print(f"{len(cell)} atoms")

(MD / "in.lammps").write_text(f"""log log.lammps.run
units metal
boundary p p p
atom_style atomic
read_data AlLi.lammps-data
pair_style grace/fs extrapolation
pair_coeff * * {fs_yaml} {fs_asi} Al Li
neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes
fix     grace_gamma all pair 10 grace/fs gamma 1
compute max_gamma all reduce max f_grace_gamma
compute dist all pair/local dist
compute min_dist all reduce min c_dist inputs local
variable maxg equal c_max_gamma
variable dump_skip equal "c_max_gamma < 1.5"
dump extrap all custom 10 extrapolative_structures.dump id type x y z f_grace_gamma
dump_modify extrap skip v_dump_skip element Al Li
fix stopper all halt 10 v_maxg > 25 error continue
thermo_style custom step temp pe etotal press vol c_min_dist c_max_gamma
thermo 25
thermo_modify flush yes
velocity all create 500.0 12345 mom yes rot yes dist gaussian
timestep 0.001
fix npt_ramp all npt temp 500 5000 0.1 aniso 0 0 1.0
run {MD_STEPS}
""")
print("in.lammps written")

In [ ]:
log_file = MD / "log.lammps.run"
if not have(log_file, "(MD already run)"):
    if not LMP.exists():
        raise FileNotFoundError("no LAMMPS binary: set LMP_DRIVE_ID in the setup cell and run it again")
    t0 = time.time()      # one rank, Kokkos/CUDA: -sf kk turns grace/fs into grace/fs/kk
    run(f"{LMP} -k on g 1 -sf kk -pk kokkos newton on neigh half -in in.lammps", cwd=MD,
        LD_LIBRARY_PATH=f"{LMP.parents[1] / 'lib'}:" + os.environ.get("LD_LIBRARY_PATH", ""))
    print(f"\nMD took {time.time()-t0:.0f}s")
performance = [l.strip() for l in open(log_file) if l.startswith("Performance:")]
print(performance[-1] if performance else "(no Performance line in the log)")

In [ ]:
def read_thermo(logfile):
    """The thermo table of a LAMMPS log as a DataFrame."""
    lines = open(logfile).read().splitlines()
    start = max(i for i, l in enumerate(lines) if l.strip().startswith("Step"))
    cols, rows = lines[start].split(), []
    for l in lines[start+1:]:
        values = l.split()
        if len(values) != len(cols): break
        try:    rows.append([float(v) for v in values])
        except ValueError: break
    return pd.DataFrame(rows, columns=cols)

thermo = read_thermo(log_file)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(thermo["Step"], thermo["Temp"]); ax[0].set_xlabel("step"); ax[0].set_ylabel("T (K)"); ax[0].set_title("temperature ramp")
ax[1].semilogy(thermo["Step"], thermo["c_max_gamma"], color="tab:red")
ax[1].axhline(1.5, ls="--", c="k", lw=1,   label="dump threshold")
ax[1].axhline(25,  ls=":",  c="k", lw=1.2, label="halt threshold")
ax[1].set_xlabel("step"); ax[1].set_ylabel(r"max per-atom $\gamma$"); ax[1].set_title("extrapolation grade during MD"); ax[1].legend()
plt.tight_layout(); plt.show()

halt = [l.strip() for l in open(log_file) if "halt condition" in l]
print(halt[0] if halt else f"completed {MD_STEPS} steps without halting")
print(f"gamma: start {thermo['c_max_gamma'].iloc[0]:.2f}  ->  end {thermo['c_max_gamma'].iloc[-1]:.2f}")

### Close the loop — `pace_select`

The dumped extrapolative frames are ranked by D-optimality; the selected structures are the ones
worth computing with DFT and folding back into training.

In [ ]:
dump_file = MD / "extrapolative_structures.dump"
if not dump_file.exists():
    print("no extrapolative frames were dumped - the run stayed in-distribution")
else:
    nframes = sum(1 for l in open(dump_file) if l.startswith("ITEM: TIMESTEP"))
    print(f"{nframes} extrapolative frames dumped ({dump_file.stat().st_size/1e6:.1f} MB)")
    selected_file = MD / "selected.pkl.gz"
    if not have(selected_file, "(selection)"):
        run(f"pace_select extrapolative_structures.dump -p {fs_yaml} -a {fs_asi} -e 'Al Li' -m 20", cwd=MD)
    selected = load(selected_file)
    print(f"selected for DFT: {len(selected)} structures")

## Summary

| stage | what it showed |
|---|---|
| data selection (`fps-all`, 128 of 1000) | the foundation model's feature space spreads the picks over compositions and strains the hull set lacks; forces of several eV/Å enter the training set |
| teacher (3L, frozen readout, 128 structures) | meV/atom accuracy on held-out DFT after a few hundred updates |
| UQ (rp-dim 128, elbow search, p99 calibration) | orders of magnitude between training γ and the held-out prototype |
| γ vs error | the relaxed structures with high γ are the ones with large hull errors |
| distillation pool | ~2000 structures over all hull compositions, forces far beyond the training set |
| student (GRACE-FS-OMAT) | a few meV/atom from the teacher, and from DFT |
| hull from relaxed structures | the teacher matches the DFT hull to about 1 meV/atom; the student is several meV/atom off, and since two DFT compositions lie within 7 meV/atom of each other that can move its minimum |
| speed-up | a C++ evaluator on one CPU core against a 3-layer TF model on the GPU |
| MD | γ rises with temperature, the run halts once out of domain, frames are selected for DFT |

The cell below collects the numbers from *this* run next to the hull-only reference run's.

**Caveats**

* Two different γ live in this notebook: the teacher's GMM γ (§5 and §7–8) and the student's
  D-optimality γ (§10 and §13). Same ranking, very different scales; never plot them on one axis.
* Stress is absent from the training data, so the NPT barostat in §13 is driven by an unfitted quantity.
* GRACE/FS is used here as a worked example of a fast student. Faster GRACE variants are expected to
  supersede it.

In [ ]:
summary = pd.Series({
    "student trained on":                                  POOL_TAG,
    "teacher training structures":                         len(train_fps),
    "teacher vs DFT, test set: E-MAE (meV/atom)":          teacher_mae[0],
    "teacher vs DFT, test set: F-MAE (meV/A)":             teacher_mae[1],
    "teacher gamma: OOD median / train median":            np.median(g_ood)/np.median(g_train),
    "pool: structures":                                    len(pool),
    "pool: largest |F|max (eV/A)":                         pool["fmax"].max(),
    "pool: extrapolative for the teacher (%)":             100*(g_pool > 1).mean(),
    "student vs teacher: E-MAE (meV/atom)":                student_vs_teacher[0],
    "student vs DFT, test set: E-MAE (meV/atom)":          student_vs_dft[0],
    "student speed-up on the test set (x)":                t_tf/t_fs,
    "MD: halted at step":                                  int(halt[0].split("step")[1].split()[0]) if halt else f"no halt in {MD_STEPS} steps",
    "MD: extrapolative frames dumped":                     nframes if dump_file.exists() else 0,
})
table = pd.concat({"hull + fps (this run)": summary, "hull only (reference)": load(REF_SUMMARY)}, axis=1)
display(table.apply(lambda col: col.map(lambda v: round(v, 2) if isinstance(v, float) else v)))